# 04 · DINOv3: frozen foundation model + light tuning
The DINOv3 route from the proposal, with results shown next to the baselines and the U-Net.

* **Stage A, `frozen`:** DINOv3 encodes the RGB of every chip once, the features are cached to disk and only a small decoder is trained. The decoder also receives all 8 spectral bands (near-infrared, SWIR, NDVI...), which DINOv3 itself does not see.
* **Stage B, `lora`:** small LoRA adapters inside DINOv3 are trained as well.

Same rules as before: Train/Val data only, 5 site folds, test set locked. Bar to beat: random forest IoU 0.704.

## Access
The weights need Meta's licence accepted on Hugging Face and a logged-in session (`notebook_login()`; the token is never stored in the notebook). Steps are in `2_Code/dinov3/HOW_TO_GET_DINOV3.md`.

In [ ]:
import sys
from pathlib import Path

CODE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / '2_Code'
for p in (CODE, CODE / 'dinov3'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import paths as P
import dino_paths as DP
import train_dinov3 as T
print(P.describe())
print('DINOv3 results:', DP.DINO_RESULTS)
print('Train/Val data :', DP.TRAIN_VAL, '(exists)' if DP.TRAIN_VAL.exists() else '(MISSING: run 00_get_data first)')

## Model and run settings

In [ ]:
MODEL_ID    = 'facebook/dinov3-vitl16-pretrain-sat493m'   # satellite-trained ViT-L (recommended, ~1.2 GB)
# MODEL_ID  = 'facebook/dinov3-vits16-pretrain-lvd1689m'  # small ViT-S alternative
MODE        = 'frozen'   # 'frozen' or 'lora'
NO_SPECTRAL = False      # True = ablation: DINOv3 RGB features only
FOLDS       = None       # None = all 5 folds, or e.g. [1, 2]
EPOCHS      = None       # None = default (frozen 40, lora 30)
RUN_TRAINING = True     # set True to train
SMOKE       = False      # True = tiny random model, tests the code without weights (results are meaningless)
MAX_CHIPS   = None       # e.g. 4 for a very quick test

## Hugging Face login and licence check (skipped in smoke mode)

In [ ]:
if not SMOKE:
    try:
        from huggingface_hub import whoami, hf_hub_download
        print('logged in as:', whoami()['name'])
        hf_hub_download(MODEL_ID, 'config.json')
        print('licence accepted: you can download', MODEL_ID)
    except Exception as e:
        print('NOT READY:', type(e).__name__, str(e)[:200])
        print('-> follow 2_Code/dinov3/HOW_TO_GET_DINOV3.md (accept the licence, then huggingface-cli login)')
else:
    print('smoke mode: no login needed')

## Training (only if `RUN_TRAINING = True`)
Progress is printed per epoch. Results go to `4_Results/DINOv3/<run name>/`; smoke runs use a temporary folder.

In [ ]:
import tempfile
RUN_DIR = None
if RUN_TRAINING:
    argv = ['--model-id', MODEL_ID, '--mode', MODE]
    if NO_SPECTRAL: argv.append('--no-spectral')
    if FOLDS: argv += ['--folds', *map(str, FOLDS)]
    if EPOCHS: argv += ['--epochs', str(EPOCHS)]
    if MAX_CHIPS: argv += ['--max-chips', str(MAX_CHIPS)]
    if SMOKE:
        tmp = Path(tempfile.mkdtemp(prefix='dinov3_smoke_'))
        argv += ['--smoke', '--out', str(tmp), '--cache-dir', str(tmp / 'cache')]
        if MODE == 'lora': argv.append('--force')
    RUN_DIR = T.run(T.parse(argv))
else:
    print('RUN_TRAINING is False: showing existing results only')

## Results

In [ ]:
from IPython.display import Markdown, Image, display
runs = [RUN_DIR] if RUN_DIR else sorted([d for d in DP.DINO_RESULTS.glob('*') if (d / 'RESULTS.md').exists()],
                                       key=lambda d: (d / 'RESULTS.md').stat().st_mtime)
if not runs:
    print('No DINOv3 results yet. Set RUN_TRAINING = True above and run the notebook again.')
else:
    print('runs found:', [r.name for r in runs])
    run = runs[-1]
    display(Markdown((run / 'RESULTS.md').read_text(encoding='utf-8')))
    for f in ('iou_per_fold.png', 'training_curves.png', 'examples.png'):
        if (run / 'figs' / f).exists():
            display(Image(filename=str(run / 'figs' / f)))

## Next
* **Frozen DINOv3 vs 0.704:** if it is close, run `MODE = 'lora'`.
* **Contribution of DINOv3:** run once with `NO_SPECTRAL = True`. If RGB-only is far below the full version, most of the skill comes from the spectral bands, as in the earlier SAM work.
* The test set is scored once, for the final chosen model only.